## Basic work flow

1) Setup encounter and define character actions and current state
2) Filter available actions based on current state
3) -
DQN model is trained
4) DQN network takes in the current state, discrete actions choices -> chooses actions and runs outcome to calculate q-value for that given set of actions
5) Then the total reward for the actions, and the error from expected and actual q-value rewards are stored in a big results table to be graphed and used as a model performance metric 
6) The training loop is epsilon-greedy strategy
7) DQN incorporates expreince replay during the training stage
8) The model is reaches a trained state

Implement the trained model to then choose the difficulty of the encounter, and select the optimized action choices based on q-value 
* From the trained model -> calculate number of rounds for each monster, and the relative difficulty of the monster
* Use a metric to see the output

Repeat for each build-monster combination 

Model basics 

* Define max number of steps for an episode 
** For each step -> 
1) Pick an action 
2) Take in an observation of the environment 
3) calculate the result of the action chosen 
-> define the new observation 
4) Calculate the reward of that action choice 

Defining the DQN network 
* Setup the model itself based on number of layers 
* input layer (linear) -> processing layers -> output layers
* The linear layer should be equal to the number of actions we have 

Implementing the model 
** Forward step: takes in the model, and the current observation 
q-values == layer 1 (takes in observation) 
output layer 2 -> processes the q-values 
layer 3 -> vully connected layer 
layer 4-> impelements the qvalues chosen 

final output: q-values for the action chosen given the specific observation

How to test out the model 
** Define the metwrok 
** apply torch.tensor library to help process it all?

During a training episode 
** Reset the environment and get a new observation 
* define a number of steps for the episode 
Run an episode, which includes:
* for loop for iterable range 
** choosing an action (either random or switch to optimized exploitation after a certain number of training rounds)
** Enact the action chosen and output 1) the next observation, 2) the reward, 3) whether it was a terminal action or not 
Break the training loop once the max number of training steps is completed 


TRAINING THE MODEL
Using experience replay 
** process the environment and observation of the environemtn 
** format the reward 
experience replay class:
** model takes in the capacity (the number of episodes stored in the memory)
** a step that adds the current data to the data storage 
** A step where it select/recalls a certain number of previous samples and previous experiences
** uses a torch.tensor to calculate the state, action, rewrd, next_state, and terminal choice 
Result -> returns the information from the previous experience 

When training the model 
* define the values of the learning and training process 

Executing the training code 
1) Call the class for the experience replay 
2) Implement the model of choice 
3) Save the rewards of every training step so the training performance can be evaluated 
4) Implement an episode of the same, in which and action is chosen and then the model is implemented
5) Use the model to estimate the q-values of the chosen step -> implement the learning strategy 
6) keep track of cumulative reward 
7) Add the outcome to the experience replay storage 
8) Print out status of the result

Implement the model and get results of the q-values for each combination of actions, and chose the action with the highest q-value 

Evaluate the model 
** look at the accumulated reward from each training session 

You can save the model in torch 

---

Outline of model 

* Establish an environment, and learning constants 
** learning rate (alpha), discount factor (gamma), exploration rate (epsilon), epsilon decay, epsilon minimum
** total number of training episodes, max number of steps per training episode

q-table 
* action size 
* state size 
* q table 

Policy : epsilon-greedy strategy 
** exploration = random selections of actions 
** epsilon decays at a certain rate, and then once it passes a threshold switch to exploitation strategy 
** exploitation then chooses the action with the highest q-value 

Learning/training loop 
** temporal difference (TD) target = the expected reward of the current action + the discounted future rewards 
** TD error= the difference between the TD target and the current q-value 
** the updated rule adjusts the q-value towards the TD target 
** elarning rate (\alpha): control how much new information affects the existing q-value 
** discount factor (\gamma): emphasizes the importance of future rewards 



def update_q_table(state, action, reward, next_state):
    best_next_action = np.argmax(q_table[next_state, :])
    td_target = reward + gamma * q_table[next_state, best_next_action]
    td_error = td_target - q_table[state, action]
    q_table[state, action] += alpha * td_error

epsilon = max(epsilon_min, epsilon * epsilon_decay)

Training the agent 
** episode loop -> run multiple episodes to allow the agent to learn from different starting postions
** step loop: for each step in an episode, the agent chooses and action -> observes the outcome -> updated the q-value 
** epsilon decay: gradually reduces exploration over time 

for episode in range(num_episodes):
    state, info = env.reset()
    done = False
    
    for step in range(max_steps):
        action = choose_action(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        
        update_q_table(state, action, reward, next_state)
        
        state = next_state
        
        if done:
            break
    
    # Decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

Testing the trained agent 
** create the environment 
** performance: monitor the total rewards to assess performance

env = gym.make("Taxi-v4", render_mode='human')

for episode in range(5):
    state,  info  = env.reset()
    done = False
    total_rewards = 0
    
    for step in range(max_steps):
        env.render()
        action = np.argmax(q_table[state, :])
        next_state, reward, terminated, truncated, info = env.step(action)
        
        total_rewards += reward
        state = next_state
        
        if done:
            print(f"Episode: {episode + 1}, Total Reward: {total_rewards}")
            break

env.close()


#----------
Defining the Deep Q network model

class -> 
1) define self
2) input layer -> linear layer -> linear layer with output dimension
3) Define the step forward
	reful -> 

using pytorch:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import gym

# Define Deep Q-Network (DQN) Model
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

With experience replay buffer 
# Define DQN Agent with Experience Replay Buffer
class DQNAgent:
    def __init__(self, state_dim, action_dim, lr, gamma, epsilon, epsilon_decay, buffer_size):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.memory = deque(maxlen=buffer_size)
        self.model = DQN(state_dim, action_dim)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return np.random.choice(self.action_dim)
        q_values = self.model(torch.tensor(state, dtype=torch.float32))
        return torch.argmax(q_values).item()

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                target = reward + self.gamma * torch.max(self.model(torch.tensor(next_state, dtype=torch.float32))).item()
            target_f = self.model(torch.tensor(state, dtype=torch.float32)).numpy()
            target_f[action] = target
            self.optimizer.zero_grad()
            loss = nn.MSELoss()(torch.tensor(target_f), self.model(torch.tensor(state, dtype=torch.float32)))
            loss.backward()
            self.optimizer.step()
        if self.epsilon > 0.01:
            self.epsilon *= self.epsilon_decay

Training the agent 
# Initialize environment and agent with Experience Replay Buffer
env = gymnasium.make('Breakout-v0')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n
agent = DQNAgent(state_dim, action_dim, lr=0.001, gamma=0.99, epsilon=1.0, epsilon_decay=0.995, buffer_size=10000)

# Train the DQN agent with Experience Replay Buffer
batch_size = 32
num_episodes = 1000
for episode in range(num_episodes):
    state = env.reset()
    total_reward = 0
    done = False
    while not done:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        agent.replay(batch_size)
    print(f"Episode: {episode + 1}, Total Reward: {total_reward}")

# Evaluate the trained agent
total_rewards = []
num_episodes_eval = 10
for _ in range(num_episodes_eval):
    state = env.reset()
    total_reward = 0
    done = False
    while not done:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        state = next_state
        total_reward += reward
    total_rewards.append(total_reward)
print(f"Average Total Reward (Evaluation): {np.mean(total_rewards)}")

----

# Gather data on each of the player and monsters, setup the environment and state, and all possible action options for both models
* Load datasets and have four data: 1) Player stats dictionary (player state) 2) enemy stats dictionary (enemy state), 3) All player actionlist, 4) all enemy action list
* reset the environment at the start of the encounter: 1) The order of iniative (who goes first), 2) The current x-value location of the player and enemy, 3) The starting stats (health, etc) of both

## Run a simulation of one encounter 
* Taken in the observation of the current state of player-enemy and filter to only the avaiable possible actions
* Implement the DQN model based on the observation and discrete actions
* Define the action-chosing policy (random vs exploitation)
* After an action is chosen, use the model to calculate the q-value and reward of that action
* Process the outcome of the action -> What was the reward? Did this meet what was calcualted (TD)?
* Define the next observation state -> player/enemy stats (health, location, apply condititions)
* Repeat process until the encounter is resolved
* Calcualte the total reward based on the outcome of the encounter (who wins / number of rounds)

## Train the DQN model based on the code for running one simulation 
* Implement 'experience replay' to help calcualte the q-values, based on previous results
* Evalaute the training performance by graphing the calcualte reward, and the difference between expected-actual reward values
* save the trained model in torch, and training encounter results in duck DB

## Implement the trained model to run 20 encounter simulations to choose the best action combinations
* caluclate the TOTAL rounds and results of taking those actions
* Use this to define encoutner difficulty
* Have Final metric be a combination of average encounter difficulty, and percent that the results were player won
* Have a final graph of each action/ action combo and the q-values of that combination against the enemy

## Implement the trained model for a given player -> use it to predict monster difficulty (monsters choose optimal actions?) 

## Implement the trained player model to the create a trained enemy model --> repeat the process?

## Final outcome for capstone presentation
* Model training performances
* Resulting difficulty of each 20-encounter simulation
* How this performs compared to actual human encoutner data